# Making cutout image using ra, dec of SN and host galaxy (only host galaxy 1) 

Uncomment it if you need to install reporject 

In [ ]:
#%pip install reproject 

In [2]:
import pandas as pd

In [10]:
hosts = pd.read_pickle("results_prost_snIa_tns.pkl")

In [11]:
hosts

,diaObjectId,tns_name,ra,dec,idx,best_cat,best_cat_release,query_time,smallcone_posterior,missedcat_posterior,...,host_2_redshift_info,host_2_absmag_mean,host_2_absmag_std,host_2_absmag_info,host_2_offset_mean,host_2_offset_std,host_2_offset_info,host_2_offset_posterior,any_posterior,none_posterior
0,170019717277810735,2026ctw,151.740293,1.927064,0,decals,dr9,1.869315,3.059538e-15,3.059538e-15,...,PHOT,-19.797121,1.215717,r,292.429866,0.106183,,0.0,1.0,9.178615e-15
1,170028486191218743,2026qrh,64.185342,-48.943229,0,decals,dr9,2.913486,1.960517e-15,1.960517e-15,...,PHOT,-21.501667,1.154754,r,297.725152,0.097636,,0.0,1.0,5.881552e-15
2,170028500610711681,2024xom,150.746125,2.336469,0,decals,dr9,2.270807,9.451034e-15,9.451034e-15,...,PHOT,-19.519884,1.182168,r,299.436545,0.096966,,0.0,1.0,2.835310e-14
3,170028510695916335,2025mbp,151.592617,1.158385,0,decals,dr9,1.709096,1.157273e-13,1.157273e-13,...,PHOT,-19.128257,1.140447,r,294.467641,0.098585,,0.0,1.0,3.471820e-13
4,170028520834072659,2026fgl,149.218165,3.469919,0,decals,dr9,1.818391,1.561005e-15,1.561005e-15,...,PHOT,-20.638615,1.168080,r,297.403899,0.097603,,0.0,1.0,4.683014e-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,170657505023098990,2026tzm,356.991032,0.193536,0,decals,dr9,2.350995,2.396926e-15,2.396926e-15,...,PHOT,-19.618029,1.200574,r,292.374203,0.098949,,0.0,1.0,7.190777e-15
121,313893021499785278,2026ezw,149.635687,1.269344,0,decals,dr9,1.806827,6.078782e-15,6.078782e-15,...,PHOT,-20.999280,1.155281,r,298.327531,0.099575,,0.0,1.0,1.823635e-14
122,313932566867476549,2026qrh,64.185342,-48.943229,0,decals,dr9,2.459991,1.925857e-15,1.925857e-15,...,PHOT,-21.501667,1.154754,r,297.719138,0.096694,,0.0,1.0,5.777571e-15
123,313972182793322538,2026fgl,149.218165,3.469919,0,decals,dr9,1.377744,1.555678e-15,1.555678e-15,...,PHOT,-20.638615,1.168080,r,297.404054,0.099627,,0.0,1.0,4.667033e-15


In [12]:
def imaging_survey(row):
    catalog = str(row["best_cat"]).lower()
    if catalog == "none":
        return "skymapper" if float(row["dec"]) < -30 else "panstarrs"
    return catalog

## 1. Making r-band cutout

In [13]:
from io import BytesIO, StringIO

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.visualization import ZScaleInterval


def get_survey_cutout(ra, dec, survey, band="r", size_arcsec=120):
    if pd.isna(survey):
        return None
    survey = survey.lower()

    if survey == "decals":
        # The Legacy Survey cutout service limits images to 512 pixels.
        size_px = min(512, round(size_arcsec / 0.262))
        pixscale = max(0.262, size_arcsec / size_px)

        response = requests.get(
            "https://www.legacysurvey.org/viewer/fits-cutout",
            params={
                "ra": ra, "dec": dec,
                "layer": "ls-dr9-south",  # DECaLS imaging used with DR9
                "bands": band,
                "size": size_px,
                "pixscale": pixscale,
            },
            timeout=60,
        )

    elif survey == "panstarrs":
        # First ask MAST which image contains this position.
        response = requests.get(
            "https://ps1images.stsci.edu/cgi-bin/ps1filenames.py",
            params={"ra": ra, "dec": dec, "filters": band},
            timeout=60,
        )
        response.raise_for_status()
        images = pd.read_csv(StringIO(response.text), sep=r"\s+")
        if images.empty:
            raise ValueError(f"No Pan-STARRS {band}-band image at {ra}, {dec}")

        response = requests.get(
            "https://ps1images.stsci.edu/cgi-bin/fitscut.cgi",
            params={
                "ra": ra, "dec": dec,
                "size": round(size_arcsec / 0.25),
                "format": "fits",
                "red": images.iloc[0]["filename"],
            },
            timeout=60,
        )

    elif survey == "skymapper":
        # SkyMapper returns a table of matching image cutouts.
        response = requests.get(
            "https://api.skymapper.nci.org.au/public/siap/dr4/query",
            params={
                "POS": f"{ra},{dec}",
                "SIZE": size_arcsec / 3600,
                "BAND": band,
                "FORMAT": "image/fits",
                "INTERSECT": "CENTER",
                "RESPONSEFORMAT": "CSV",
            },
            timeout=60,
        )
        response.raise_for_status()
        images = pd.read_csv(StringIO(response.text))
        if images.empty:
            raise ValueError(f"No SkyMapper {band}-band image at {ra}, {dec}")

        response = requests.get(images.iloc[0]["get_fits"], timeout=60)

    else:
        raise ValueError(
            "Choose 'decals', 'panstarrs', or 'skymapper'. "
            "GLADE+ has no dedicated image cutout."
        )

    response.raise_for_status()

    with fits.open(BytesIO(response.content)) as hdul:
        image_hdu = next(
            hdu for hdu in hdul
            if hdu.data is not None and hdu.data.ndim == 2
        )
        data = image_hdu.data.astype(float)
        wcs = WCS(image_hdu.header).celestial

    return data, wcs


def plot_prost_host_survey(row, survey, band="r", size_arcsec=120):
    ra, dec = float(row["ra"]), float(row["dec"])
    data, wcs = get_survey_cutout(ra, dec, survey, band, size_arcsec)

    if survey is None:
        return None

    finite = data[np.isfinite(data)]
    if finite.size == 0:
        raise ValueError("The returned cutout has no finite image pixels")
    vmin, vmax = ZScaleInterval().get_limits(finite)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(data, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)

    def mark(ra_value, dec_value, label, marker, color, size):
        if pd.notna(ra_value) and pd.notna(dec_value):
            x, y = wcs.world_to_pixel(
                SkyCoord(float(ra_value), float(dec_value), unit="deg")
            )
            ax.plot(
                x, y, marker=marker, linestyle="none",
                color=color, markersize=size, mew=2,
                markerfacecolor="none" if marker == "o" else color,
                label=label,
            )

    mark(ra, dec, "Transient", "+", "cyan", 8)
    mark(row["host_ra"], row["host_dec"], "PROST host", "o", "red", 8)
    #mark(row["host_2_ra"], row["host_2_dec"], "PROST host 2", "x", "orange", 8)

    ax.set(
        xlim=(0, data.shape[1]),
        ylim=(0, data.shape[0]),
        xlabel="Cutout x (pixels)",
        ylabel="Cutout y (pixels)",
        title=f'{row["tns_name"]} — {survey} {band}-band',
    )
    ax.legend()
    return fig

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

output_dir = Path("./cutout_image_rband")
output_dir.mkdir(parents=True, exist_ok=True)

for _, row in hosts.iterrows():
    survey = imaging_survey(row)

    fig = plot_prost_host_survey(row, survey=survey)

    fig.savefig(
        output_dir / f"prost_host_{row['tns_name']}_{survey}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.close(fig)

## 2. Making rgb composite cutout (optional)
But some of case, if they have bad va

In [ ]:
from astropy.visualization import make_lupton_rgb
from reproject import reproject_interp


def plot_prost_host_rgb(row, survey, size_arcsec=120, Q=8, stretch=0.5):
    if pd.isna(survey):
        return None

    survey = survey.lower()
    ra, dec = float(row["ra"]), float(row["dec"])

    # Ordered from blue to red
    bands = ("g", "r", "z") if survey == "decals" else ("g", "r", "i")

    cutouts = {
        band: get_survey_cutout(
            ra, dec, survey, band=band, size_arcsec=size_arcsec
        )
        for band in bands
    }

    # Align every image to the r-band pixel grid.
    reference, wcs = cutouts["r"]
    shape = reference.shape
    aligned = {}
    valid = np.ones(shape, dtype=bool)

    for band in bands:
        data, band_wcs = cutouts[band]

        if band == "r":
            image = data.copy()
            footprint = np.ones(shape)
        else:
            image, footprint = reproject_interp(
                (data, band_wcs),
                wcs,
                shape_out=shape,
            )

        good = np.isfinite(image) & (footprint > 0)
        if not good.any():
            raise ValueError(f"No valid pixels in {band}-band cutout")

        valid &= good

        # Background subtraction and per-band scaling for visualization.
        background = np.median(image[good])
        image = image - background
        scale = np.percentile(image[good], 99.5)

        if scale <= 0:
            raise ValueError(f"No usable signal in {band}-band cutout")

        aligned[band] = np.where(
            good, np.clip(image / scale, 0, None), 0
        )

    rgb = make_lupton_rgb(
        aligned[bands[2]],  # Red: i or z
        aligned["r"],      # Green: r
        aligned["g"],      # Blue: g
        Q=Q,
        stretch=stretch,
    )
    rgb[~valid] = 0

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(rgb, origin="lower")

    def mark(ra_key, dec_key, label, marker, color, size):
        ra_value = row.get(ra_key, np.nan)
        dec_value = row.get(dec_key, np.nan)

        if pd.notna(ra_value) and pd.notna(dec_value):
            coord = SkyCoord(
                float(ra_value), float(dec_value), unit="deg"
            )
            x, y = wcs.world_to_pixel(coord)
            ax.plot(
                x, y,
                marker=marker,
                linestyle="none",
                color=color,
                markersize=size,
                markeredgewidth=1.5,
                markerfacecolor="none",
                label=label,
            )

    mark("ra", "dec", "Transient", "+", "cyan", 12)
    mark("host_ra", "host_dec", "PROST host", "o", "red", 12)

    ax.set(
        xlim=(-0.5, shape[1] - 0.5),
        ylim=(-0.5, shape[0] - 0.5),
        xlabel="Cutout x (pixels)",
        ylabel="Cutout y (pixels)",
        title=f'{row["tns_name"]} — {survey} {"".join(bands)}',
    )
    ax.legend()
    fig.tight_layout()
    return fig

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

output_dir = Path("./cutout_image_color_composite")
output_dir.mkdir(parents=True, exist_ok=True)

for _, row in hosts.iterrows():
    survey = imaging_survey(row)
    print(survey)

    fig = plot_prost_host_rgb(row, survey, size_arcsec=120)

    fig.savefig(
        output_dir / f"prost_host_{row['tns_name']}_{survey}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.close(fig)